In [1]:
import gymnasium as gym
from stable_baselines3 import PPO,HerReplayBuffer, SAC
import highway_env
from highway_carla_wrappers import *
from carla_parking import SimulationParking


In [2]:
simulation = SimulationParking()
simulation.load_world("Town05")
initial_location = {
    "x": 20,
    "y": -30,
    "z": 0.3,
    "yaw": 180,
}
vehicle = simulation.init_vehicle("model3", initial_location)
goal_corners = [
    simulation.get_location_by_coordinates(6, -28.5, 0),
    simulation.get_location_by_coordinates(6, -31.5, 0),
    simulation.get_location_by_coordinates(11.5, -28.5, 0),
    simulation.get_location_by_coordinates(11.5, -31.5, 0)
]
simulation.init_spectator()

In [3]:

env = gym.make("parking-v0", render_mode="human")

env = CarlaInitRoadWrapper(env, carla_client=simulation, carla_vehicle=vehicle)
env = CarlaObservationWrapper(env, carla_client=simulation)
env = CarlaActionWrapper(env, carla_client=simulation, carla_vehicle=vehicle)

# SAC hyperparams:
model = SAC(
    "MultiInputPolicy",
    env,
    replay_buffer_class=HerReplayBuffer,
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy="future",
    ),
    verbose=1,
    learning_starts=1000,
    buffer_size=int(1e6),
    learning_rate=1e-3,
    gamma=0.95,
    batch_size=256,
    policy_kwargs=dict(net_arch=[256, 256, 256]),
)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [4]:
model.learn(int(1e6))
model.save('her_sac_highway_carla')


Goal creado en posición: [  8.75 -30.  ], heading: 0.0°
Post-reset, lanes del Road actual:
('a', 'b', 0): start=[  6. -30.], end=[ 11.5 -30. ]
Applied action to CARLA: throttle=0.0, steering=0.6456120014190674
Applied action to CARLA: throttle=0.0, steering=0.9395723342895508
Applied action to CARLA: throttle=0.22491776943206787, steering=-0.911412239074707
Applied action to CARLA: throttle=0.6996753215789795, steering=0.11940503120422363
Applied action to CARLA: throttle=0.944866418838501, steering=0.5852494239807129
Applied action to CARLA: throttle=0.0, steering=-0.9110574126243591
Applied action to CARLA: throttle=0.6874761581420898, steering=-0.7040848135948181
Applied action to CARLA: throttle=0.6296813488006592, steering=-0.0545806884765625
Applied action to CARLA: throttle=0.9902229309082031, steering=0.6067495346069336
Applied action to CARLA: throttle=0.8752079010009766, steering=-0.7622573971748352
Applied action to CARLA: throttle=0.3862797021865845, steering=0.198385477066

KeyboardInterrupt: 

In [ ]:
model = SAC.load('her_sac_highway_carla', env=env)


In [ ]:
# env = gym.make("parking-v0", render_mode="human")

obs, _ = env.reset()

# Evaluate the agent
episode_reward = 0
for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    done = truncated or terminated
    episode_reward += reward
    if done or info.get("is_success", False):
        print("Reward:", episode_reward, "Success?", info.get("is_success", False))
        episode_reward = 0.0
        obs, _ = env.reset()